#### What we're doing

This is a HTET script for a 0+1D scalar field theory (quantum mechanics)

We use:

$$H = \frac{1}{2}\Pi^2 + \frac{1}{2}\phi^2 + \lambda \phi^4 $$

Free part is just harmonic oscillator and $\lambda$ term is the interaction term

Imports

In [2]:
import numpy as np

#### Annihilation and Creation Operators

```N``` is the number of truncated states

standard ladder operators for fock basis {$\ket{0}, \ket{1}, \ket{2},...\ket{N-1},$}

$ a\ket{n} = \sqrt{n}\ket{n-1} ~~~~ a^{\dagger} \ket{n} = \sqrt{n+1}\ket{n+1} $ 

In [3]:
def ladder_ops(N: int):
    a = np.zeros((N, N), dtype=float)
    for n in range(1, N):
        a[n - 1, n] = np.sqrt(n)
    adag = a.T
    return a, adag


#### Construct $\phi$ and $\Pi$

```omega``` is the input frequency

$$\phi = \frac{a+a^{\dagger}}{\sqrt{2\omega}}$$

$$\Pi = \sqrt{\frac{\omega}{2}} (a^{\dagger}-a) $$

In [4]:
def ho_operators(N: int, omega: float):
    a, adag = ladder_ops(N)
    phi = (a + adag) / np.sqrt(2.0 * omega)
    pi = 1j * np.sqrt(omega / 2.0) * (adag - a)
    return phi, pi, a, adag


#### Full Hamiltonian

This is the free and interacting parts of the hamiltonian in the HO basis

In [5]:
def build_H0(N: int, omega: float):
    n = np.arange(N, dtype=float)
    return np.diag(omega * (n + 0.5))


def build_V_x4(N: int, omega: float, lam: float):
    x, _, _, _ = ho_operators(N, omega)
    x4 = x @ x @ x @ x
    return (lam / 24.0) * x4


def build_full_H(N: int, omega: float, lam: float):
    return build_H0(N, omega) + build_V_x4(N, omega, lam)

##### Defines the UV cutoff on the Hamiltonian

- Assumes HO free Hamiltonian
- Uses known analytic spectrum
- splits basis into low energy and high energy sectors P and Q respectively

inputs:
```omega``` is the input frequency 

```Emax``` is the UV cutoff

```N_uv``` is the regulator of the space size not the HTET cutoff

In [6]:
def indices_below_Emax(omega: float, Emax: float, N_uv: int):
    n = np.arange(N_uv)
    En = omega * (n + 0.5)
    P = np.where(En <= Emax)[0]
    Q = np.where(En > Emax)[0]
    return P, Q, En

#### Building the effective Hamiltonian

The Hamiltonian constructed here acts only within the low energy space P but includes effects of the excluded high energy space Q through the HTET matching correction at $O(V^2)$:

$$H_{eff}=P(H_0+V)P + H_2 ~~~~~ (H_2)_{fi} = \sum_{\alpha \in Q} \frac{V_{f\alpha}V_{\alpha i}}{E_f - E_\alpha}$$

THe function returns (Heff, P, Q) where Heff is the P space effective Hamiltonian matched through O(V^2) using Cohen 2.23b.

Also in general this is non hermitian which is expected from Cohen

In [7]:
def htet_Heff_O_V2(N_uv: int, omega: float, lam: float, Emax: float):
    
    #build H0 and V in the truncated basis
    #gives the "fundamental data" needed for matching
    H0 = build_H0(N_uv, omega)
    V = build_V_x4(N_uv, omega, lam)

    P, Q, En = indices_below_Emax(omega, Emax, N_uv)
    nP = len(P)
    #checking we actually have P states to use
    if nP == 0:
        raise ValueError("Emax too small: P-space is empty.")

    # Project to blocks
    # our original is built from H0_PP and V_PP
    # our correction is built from V_PQ and V_QP
    H0_PP = H0[np.ix_(P, P)] #P H0 P
    V_PP = V[np.ix_(P, P)] #P V P
    V_PQ = V[np.ix_(P, Q)] #P V Q
    V_QP = V[np.ix_(Q, P)] # Q V P

    # We build the "naive truncation" (PHP)
    Heff = H0_PP + V_PP

    # set up construction of H2
    H2 = np.zeros((nP, nP), dtype=complex)

    #pull out the H0 energies for P and Q (for denominators)
    EP = En[P]  # energies for P states (H0) -  EP[f] is the free energy of the P state |f>
    EQ = En[Q]  # energies for Q states (H0) - EP[a] is the freee energy of the Q state |a>

    # Compute correction row by row
    for f in range(nP):
        Ef = EP[f] # fix final p state |f>
        inv = 1.0 / (Ef - EQ) # build denominators
        H2[f, :] = (V_PQ[f, :] * inv) @ V_QP # construct correction

    Heff = Heff + H2
    return Heff, P, Q # theres the full effective hamiltonian :)

#### GS energy

Computes eigenvalues of of H then gives the minimum 

In [8]:
def ground_state_energy_from_matrix(H):
    evals, vecs = np.linalg.eig(H) # extracts jth eigenvalue and coresponding eigenvector
    weights = np.abs(vecs[0, :])**2 # extracts overlap of each eigenvector with HO ground state
    j = np.argmax(weights) # finds index of eigenvector with largest overlap
    return evals[j], evals # returns eigenvalue associated with the physically relevant ground state and all eigenvalues

#we do this complicated thing because if we dont we end up with majorly negative energies


##### Run of functions and raw comparison

In [9]:
# default run
def run_scan(omega=1.0,lam=1.0,N_uv=200,Emax_list=(6.0, 7.0, 8.0, 9.0, 10.0),): 
    H_full = build_full_H(N_uv, omega, lam) #builds the Hamiltonian in truncated basis

    #prints info and table headers
    print(f"omega={omega}  lambda(Cohen)={lam}  N_uv={N_uv}")
    print("Emax | dim(P) | E0_raw (Re,Im) | E0_HTET (Re,Im)")
    print("-" * 70)

    #loops over each Emax
    for Emax in Emax_list:
        #defines P,Q,En for given Emax
        P, Q, En = indices_below_Emax(omega, Emax, N_uv)
        nP = len(P)

        # sets up a raw truncation to benchmark
        H_raw = H_full[np.ix_(P, P)]
        E0_raw, _ = ground_state_energy_from_matrix(H_raw)

        # HTET through O(V^2)
        H_htet, _, _ = htet_Heff_O_V2(N_uv, omega, lam, Emax)
        E0_htet, _ = ground_state_energy_from_matrix(H_htet)

        #prints results
        print(f"{Emax:4.1f} | {nP:6d} | "f"{E0_raw.real: .10f},{E0_raw.imag: .2e} | "f"{E0_htet.real: .10f},{E0_htet.imag: .2e}")

In [10]:
run_scan(
    omega=1.0,
    lam=32.0,
    N_uv=250,
    Emax_list=[6.0, 7.0, 8.0, 9.0, 10.0, 12.0, 50.0],
    )


omega=1.0  lambda(Cohen)=32.0  N_uv=250
Emax | dim(P) | E0_raw (Re,Im) | E0_HTET (Re,Im)
----------------------------------------------------------------------
 6.0 |      6 |  0.8699766826, 0.00e+00 |  0.8635398432, 0.00e+00
 7.0 |      7 |  0.8621707590, 0.00e+00 |  0.8517061238, 0.00e+00
 8.0 |      8 |  0.8621707590, 0.00e+00 |  0.8517061238, 0.00e+00
 9.0 |      9 |  0.8621687949, 0.00e+00 |  0.8607267974, 0.00e+00
10.0 |     10 |  0.8621687949, 0.00e+00 |  0.8607267974, 0.00e+00
12.0 |     12 |  0.8613851931, 0.00e+00 |  0.8621557497, 0.00e+00
50.0 |     50 |  0.8597426908, 0.00e+00 |  0.8597426928, 0.00e+00


In [11]:
run_scan(
    omega=1.0,
    lam=1,
    N_uv=250,
    Emax_list=[6.0, 7.0, 8.0, 9.0, 10.0, 12.0, 50.0],
    )


omega=1.0  lambda(Cohen)=1  N_uv=250
Emax | dim(P) | E0_raw (Re,Im) | E0_HTET (Re,Im)
----------------------------------------------------------------------
 6.0 |      6 |  0.5277631488, 0.00e+00 |  0.5276599832, 0.00e+00
 7.0 |      7 |  0.5277363427, 0.00e+00 |  0.5277329516,-4.51e-19
 8.0 |      8 |  0.5277363427, 0.00e+00 |  0.5277329516,-5.67e-19
 9.0 |      9 |  0.5277361912, 0.00e+00 |  0.5277359274,-3.21e-18
10.0 |     10 |  0.5277361912, 0.00e+00 |  0.5277359274,-4.76e-18
12.0 |     12 |  0.5277361368, 0.00e+00 |  0.5277346880, 0.00e+00
50.0 |     50 |  0.5277361273, 0.00e+00 |  0.5277361273, 0.00e+00
